In [1]:
import torch
import pandas as pd

torch.manual_seed(42)

dataframe = pd.read_csv('./coin_Bitcoin.csv')

In [14]:
# Check if CUDA is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(dataframe.shape)

Using device = cpu
(2991, 10)


In [4]:
import sklearn
from sklearn.preprocessing import StandardScaler

# take a look at the csv file yourself first
# columns High, Low, Open are input features and column Close is target value
x = dataframe[['High', 'Low', 'Open']]
y = dataframe[['Close']]

scaler_x = StandardScaler(copy=True)
scaler_y = StandardScaler(copy=True)

# use StandardScaler from sklearn to standardize
x = scaler_x.fit_transform(x)
y = scaler_y.fit_transform(y)

In [5]:
from sklearn.model_selection import train_test_split

# split into train and evaluation (8 : 2) using train_test_split from sklearn
train_x, test_x, train_y, test_y = train_test_split(x, y, train_size = 0.8)

In [10]:
print(test_x.shape)

(599, 3)


In [15]:
# now make x and y tensors, think about the shape of train_x, it should be (total_examples, sequence_lenth, feature_size)
# we wlll make sequence_length just 1 for simplicity, and you could use unsqueeze at dimension 1 to do this
# also when you create tensor, it needs to be float type since pytorch training does not take default type read using pandas
train_x = torch.tensor(data=train_x, dtype=torch.float32)
train_y = torch.tensor(data=train_y, dtype=torch.float32)
test_x = torch.tensor(data=test_x, dtype=torch.float32)
seq_len = train_x[0].shape[0] # it is actually just 1 as explained above

/usr/local/lib/python3.12/dist-packages/torch/utils/_device.py:109: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return func(*args, **kwargs)


In [16]:
from torch.utils.data import Dataset

# different from CNN which uses ImageFolder method, we don't have such method for RNN, so we need to write the dataset class ourselves, reference tutorial is in the main documentation
class BitCoinDataSet(Dataset):
    def __init__(self, train_x, train_y):
        super(Dataset, self).__init__()
        self.x = train_x
        self.y = train_y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [46]:
from torch.utils.data import DataLoader

# now prepare the dataloader for training set and evaluation set, and hyperparameters
hidden_size = 256
num_layers = 2
learning_rate = 0.001
batch_size = 32
epoch_size = 10

train_dataset = BitCoinDataSet(train_x, train_y)
test_dataset = BitCoinDataSet(test_x, test_y)

train_loader = DataLoader(train_dataset,
                          batch_size=batch_size,
                          shuffle=True,
                          num_workers=2)

test_loader = DataLoader(test_dataset,
                         batch_size = batch_size,
                         shuffle=False,
                         num_workers=2)

In [47]:
import torch.nn as nn

# model design goes here
class RNN(nn.Module):

    # there is no "correct" RNN model architecture for this lab either, you can start with a naive model as follows:
    # lstm with 5 layers (or rnn, or gru) -> linear -> relu -> linear
    # lstm: nn.LSTM (https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html)

    def __init__(self, input_feature_size, hidden_size, num_layers):
        super(RNN, self).__init__()
        self.lstm = nn.LSTM(input_size=input_feature_size,
                            hidden_size = hidden_size,
                            num_layers=num_layers,
                            batch_first=True)
        self.linear1 = nn.Linear(in_features=hidden_size, out_features=256)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(in_features=256, out_features=1)

    def forward(self, x):
        x = x.unsqueeze(1)
        out, hidden = self.lstm(x)
        out = out[:, -1, :]
        out = self.linear1(out)
        out = self.relu(out)
        out = self.linear2(out)
        return out

In [48]:
from torch.optim import Adam

# instantiate your rnn model and move to device as in cnn section
rnn = RNN(3, hidden_size, num_layers).to(device)
# loss function is nn.MSELoss since it is regression task
criteria = nn.MSELoss()
# you can start with using Adam as optimizer as well
optimizer = Adam(params=rnn.parameters(),
                 lr=learning_rate)

In [49]:
# start training
rnn.train()
for epoch in range(epoch_size): # start with 10 epochs

    loss = 0.0 # you can print out average loss per batch every certain batches

    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # get inputs and target values from dataloaders and move to device
        inputs = inputs.to(device)
        targets = targets.to(device)

        # zero the parameter gradients using zero_grad()
        optimizer.zero_grad()

        # forward -> compute loss -> backward propogation -> optimize (see tutorial mentioned in main documentation)
        pred = rnn(inputs)
        batch_loss = criteria(pred, targets)
        batch_loss.backward()
        optimizer.step()

        loss += batch_loss# add loss for current batch
        if batch_idx % 100 == 99:    # print average loss per batch every 100 batches
            print(f'[{epoch + 1}, {batch_idx + 1:5d}] loss: {loss / 100:.3f}')
            loss = 0.0

print('Finished Training')

Finished Training


In [51]:
prediction = []
ground_truth = []
# evaluation
rnn.eval()
with torch.no_grad():
    for (inputs, targets) in test_loader:
        inputs = inputs
        targets = targets
        inputs = inputs.to(device)

        ground_truth += targets.flatten().tolist()
        out = rnn(inputs).detach().cpu().flatten().tolist()
        prediction += out

In [57]:
import numpy as np
from sklearn.metrics import r2_score

prediction = np.array(prediction).reshape(-1, 1)
ground_truth = np.array(ground_truth).reshape(-1, 1)

# remember we standardized the y value before, so we must reverse the normalization before we compute r2score
prediction = scaler_y.inverse_transform(prediction, copy=True)
ground_truth = scaler_y.inverse_transform(ground_truth, copy=True)

# use r2_score from sklearn
r2score = r2_score(ground_truth, prediction)
print(r2score)

0.9983289000454179
